In [ ]:
import pandas as pd

# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_FILE        = 'full_df_binary_labels.csv'   # ← change to your actual filename
OUTPUT_FILE       = 'shared_sample_200k.csv'
SHARED_SAMPLE_SIZE = 200_000
RANDOM_STATE      = 42
LABEL_COL         = 'Label'                        # ← change if your label column has a different name

# ── LOAD ──────────────────────────────────────────────────────────────────────
print('=' * 60)
print('Loading dataset...')
print('=' * 60)

# 16M rows is large — read in chunks to avoid memory issues
chunk_list = []
chunk_size = 500_000

for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=chunk_size)):
    chunk_list.append(chunk)
    rows_so_far = (i + 1) * chunk_size
    print(f'   Read ~{rows_so_far:,} rows...', end='\r')

full_df_binary_labels = pd.concat(chunk_list, ignore_index=True)
print(f'\n✅ Full dataset loaded: {full_df_binary_labels.shape[0]:,} rows x {full_df_binary_labels.shape[1]} columns')

# ── CHECK CLASS BALANCE BEFORE SAMPLING ───────────────────────────────────────
print('\n📊 Class distribution (full dataset):')
counts = full_df_binary_labels[LABEL_COL].value_counts()
ratios = full_df_binary_labels[LABEL_COL].value_counts(normalize=True)
for label in counts.index:
    print(f'   {label}: {counts[label]:,} rows ({ratios[label]*100:.2f}%)')

# ── SAMPLE ────────────────────────────────────────────────────────────────────
print(f'\n🎯 Sampling {SHARED_SAMPLE_SIZE:,} rows (stratified)...')

shared_sample = full_df_binary_labels.sample(
    n=SHARED_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
    weights=full_df_binary_labels[LABEL_COL].map(      # stratified by class
        full_df_binary_labels[LABEL_COL]
        .value_counts(normalize=True)
        .rdiv(1)                                        # inverse frequency weight
    )
).reset_index(drop=True)

# ── VERIFY CLASS BALANCE AFTER SAMPLING ───────────────────────────────────────
print('\n📊 Class distribution (sampled dataset):')
counts_s = shared_sample[LABEL_COL].value_counts()
ratios_s = shared_sample[LABEL_COL].value_counts(normalize=True)
for label in counts_s.index:
    print(f'   {label}: {counts_s[label]:,} rows ({ratios_s[label]*100:.2f}%)')

# ── SAVE ──────────────────────────────────────────────────────────────────────
print(f'\n💾 Saving to {OUTPUT_FILE}...')
shared_sample.to_csv(OUTPUT_FILE, index=False)
print(f'✅ Saved: {shared_sample.shape[0]:,} rows x {shared_sample.shape[1]} columns')
print(f'\n📋 Now change the load line in every notebook to:')
print(f'   full_df_binary_labels = pd.read_csv("{OUTPUT_FILE}")')
print('=' * 60)